# Resources
1. Stanford Precision Medicine Medical Imaging Course
2. 

# Getting Requirements


# Downloading with pip

# Importing

In [ ]:
#Preview Images
import os
import random
import matplotlib.pyplot as plt

#Data Generator
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Previewing Images

In [ ]:
# Location of images
path = 'data/train/Normal/'
image_files = os.listdir(path)

print('Sample Images')

# Set the size of the images
plt.figure(figsize=(10,5))

# Select and display random images
for i in range(4):
    plt.subplot(2, 2, i + 1)
    img = plt.imread(os.path.join(path, image_files[random.randrange(0, len(image_files))]))
    plt.imshow(img, cmap='gray')
    plt.axis('off')

# Adjust subplot parameters so that images are more evenly padded
plt.tight_layout()

# Locating and Preparing Images for Training

In [ ]:
# Normalize images: normalization typically entails scaling values to between 0 and 1.
                  # Images are made of pixels with varying intensities from 0 to 255.
                  # So normalizing the images (dividing each pixel by 255) scales the pixel values to between 0 and 1.
                  # This range of values is more effective for a neural network to learn from.
train_data_generator = ImageDataGenerator(rescale=1/255)

# Indicate the location of our images
train_folder = "data/train"

# Provide images to the machine in batches using the train_data_generator
train_generator = train_data_generator.flow_from_directory(
        train_folder,  # This indicates where images are located
                        # It contains folders, whose names will be the class labels, that contain the images
        target_size=(150, 150), # All images will be resized to 150x150
        batch_size=100,        # The generator will provide the model with 100 images at a time as it's "learning"
        class_mode="binary")   # We use "binary" because there are two classes (i.e. normal, pneumonia)


# Image Processing Pipeline
Grayscale

In [ ]:
#grayscale

# Building CNN Model Architecture

<div class="alert alert-block alert-info">
    <b>Note:</b> The below model uses Conv2D (Convolutional) layers to learn to extract the crucial features related to the defective and not defective pin. The model uses Conv2D layers because we are training on segmented bucket pin images which are two dimensional. 
</div>


In [ ]:
from tensorflow.keras import models, layers
#update with only 1 channel as gray scale to better detect features
model = models.Sequential([
    # The input shape is the shape of all images (150x150); there are 3 color channels (RGB)
    # Layer 1
    layers.Conv2D(16, (3,3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D(),

    # Layer 2
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    # Layer 3
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    # Layer 4
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    # Layer 5
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    # Flatten (combine) the learned features for prediction
    layers.Flatten(),
    layers.Dense(256, activation='relu'), # 'relu' helps the model focus on what's most important and speeds up training
    layers.Dense(1, activation='sigmoid') # 'sigmoid' ensures that the prediction will be between 0 and 1,
                                            # interpreted as the probability of being in the positive class.
])


# 'binary_crossentropy' (this is a binary task) tells the model how to measure its loss (error)
# 'adam' indicates how the model will adjust it weights, while learning, to decrease its loss (error)
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics = ['accuracy'])

# Training the Model

Start training the model to see results

In [ ]:
history = model.fit(
      train_generator,
      steps_per_epoch=10, # The total number of batches (of 100) required to "see" all 1000 of the training images.
      epochs=10)          # The number of times the model will get to go through all of the images.

# Understand the output above

As the model begins training, it receives 100 random images at a time (we previously set the batch size to 100). It then makes predictions on those 100 images. It compares its predictions to the actual labels, notes its error, and then adjusts its internal parameters (weights) to reduce its error. It is attempting to learn the best features that will enable it to correctly classify the images.

It then will receive the next batch of 100 images. This continues until the 1,000 images are exhausted.

We have set the number of epochs to 10. It means that the model can go through all 1,000 images ten times, learning as it goes. Each line is an epoch. The next line is the model going through all 1,000 of the images again (100 at a time).

Note the **accuracy** on the far right. This indicates the proportion of the model's predictions that were classified correctly. This should improve with each epoch (loop through all of the images). The model's **loss** indicates its error when making predictions. As loss drops, accuracy increases. Accuracy is probably a fair metric to use here as the dataset is balanced (500 in each class). If it were imbalanced, precision or recall metrics would prove more useful.

<div class="alert alert-block alert-info">
    <b>Note:</b> The model above is being evaluated only on the training set of images. Typically, the performance on a validation set of images (additional images the model had not previously been trained on) would also be monitored during this process. This will be demonstrated in the dedicated deep learning module.
</div>

## 3. Evaluate the model's performance on test images

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np
import os

#  The location of our test (unseen) images, used to evaluate the model's predictions
test_images = os.listdir("data/test")

# Make predictions on the test images
for file_name in test_images:
    path = "data/test/" + file_name
    test_img = image.load_img(path, target_size = (150,150))  # Resize images to the same size the model trained on
    img = image.img_to_array(test_img)
    img /= 255.0    # Normalize the images
    img = np.expand_dims(img, axis = 0)

  # Predict an image's class
    prediction = model.predict(img)

  # Output the prediction score
    print(f"Prediction: {prediction[0]}") # The model's probability score that the image is "Pneumonia"
                                        # Probability < 0.5 predicts "Normal"; .05 or greater predicts "Pneumonia"
  # Output the file name
    if prediction[0] < 0.5:
        print(file_name + " is normal")

    else:
        print(file_name + " is pneumonia")
    print('\n')